# 01 RAG系统 (Retrieval-Augmented Generation)

## 1. 概述

### 学习目标

完成本notebook后，你将能够：

1. **理解RAG** - 掌握检索增强生成的原理和优势
2. **文本嵌入** - 使用嵌入模型将文本映射到向量空间
3. **向量检索** - 构建高效的向量索引和相似度搜索
4. **文档分块** - 实现多种分块策略并理解其影响
5. **检索策略** - 掌握稠密、稀疏和混合检索方法
6. **RAG流水线** - 构建完整的检索-增强-生成流水线
7. **RAG评估** - 使用标准指标评估RAG系统质量
8. **高级技术** - 了解Self-RAG、查询扩展等前沿方法

### 前置要求

- 完成 Module 3（Transformer架构）
- 完成 Module 5（微调技术）
- 理解文本嵌入和相似度计算

### 内容结构

```
Section 3: RAG基础 .............. 动机、架构、RAG vs 纯生成
Section 4: 嵌入与向量检索 ....... 嵌入模型、FAISS索引、相似度搜索
Section 5: 文档处理与分块 ....... 分块策略、元数据、过滤检索
Section 6: 检索策略 ............. BM25、稠密检索、混合检索、重排序
Section 7: RAG流水线 ............ 上下文压缩、Prompt构建、引用生成
Section 8: RAG评估 .............. Recall@K、NDCG、ROUGE、忠实度
Section 9: 高级技术 ............. 查询扩展、Self-RAG、生产部署
```

## 2. 为什么需要RAG？

| 问题 | 纯LLM | RAG解决方案 |
|------|--------|------------|
| **幻觉** | 编造不存在的事实 | 基于检索到的真实文档回答 |
| **知识过时** | 训练数据有截止日期 | 实时检索最新文档 |
| **领域知识** | 通用知识，缺乏专业深度 | 检索领域专业文档 |
| **不可验证** | 无法追溯答案来源 | 提供引用和来源链接 |
| **更新成本** | 需要重新训练模型 | 只需更新文档库 |

> ⏱️ **预计时间**: 3-4小时 | 📊 **难度**: ⭐⭐⭐⭐ 高级

In [ ]:
# === RAG Systems - Setup ===
import numpy as np
import json
import time
import hashlib
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("✓ Libraries imported")
print(f"  NumPy: {np.__version__}")
print(f"\n📌 This notebook covers RAG (Retrieval-Augmented Generation):")
print(f"   - RAG fundamentals and motivation")
print(f"   - Text embeddings and vector search")
print(f"   - Document chunking strategies")
print(f"   - Dense, sparse, and hybrid retrieval")
print(f"   - End-to-end RAG pipeline")
print(f"   - RAG evaluation metrics")
print(f"   - Advanced RAG techniques and production")

## 3. RAG基础理论

### 3.1 什么是RAG？

RAG (Retrieval-Augmented Generation) 将信息检索与文本生成结合：

```
传统生成:  问题 → LLM → 回答（可能幻觉）

RAG生成:   问题 → 检索相关文档 → 文档+问题 → LLM → 回答（有据可查）
```

### 3.2 RAG数学形式

$$P(y|x) = \sum_{d \in \text{Top-K}} P(y|x, d) \cdot P(d|x)$$

- $x$: 用户查询
- $d$: 检索到的文档
- $y$: 生成的回答
- $P(d|x)$: 文档与查询的相关性
- $P(y|x,d)$: 基于文档生成回答的概率

### 3.3 RAG vs 纯生成 vs 微调

| 维度 | 纯生成 | 微调 | RAG |
|------|--------|------|-----|
| **知识更新** | 需要重训练 | 需要重训练 | 更新文档即可 |
| **幻觉风险** | 高 | 中 | 低（有来源） |
| **可溯源** | ✗ | ✗ | ✓（引用来源） |
| **成本** | 低 | 高（训练） | 中（检索+生成） |
| **领域适配** | 差 | 好 | 好 |
| **实时性** | 差 | 差 | 好 |

### 3.4 RAG核心组件

```
┌─────────────────────────────────────────┐
│              RAG System                  │
│                                          │
│  ┌──────────┐  ┌──────────┐  ┌────────┐ │
│  │ Indexing  │  │Retrieval │  │Generate│ │
│  │          │  │          │  │        │ │
│  │ 文档分块  │→│ 向量检索  │→│ LLM生成│ │
│  │ 嵌入生成  │  │ 重排序   │  │ 引用标注│ │
│  │ 向量存储  │  │ 过滤    │  │        │ │
│  └──────────┘  └──────────┘  └────────┘ │
└─────────────────────────────────────────┘
```

In [ ]:
# 🔬 Micro Practice: RAG vs Pure Generation
# Goal: Demonstrate why RAG is needed - compare hallucination vs grounded answers
# Expected outcome: Understand the value of retrieval-augmented generation

class PureGenerator:
    """Simulates pure LLM generation (prone to hallucination)"""
    
    def __init__(self):
        # Simulated "model knowledge" - incomplete and sometimes wrong
        self.knowledge = {
            "transformer": "Transformer是2017年提出的模型，可能用于图像分类。",
            "bert": "BERT是一个语言模型，大概有1000亿参数。",  # Wrong: 110M/340M
            "gpt": "GPT是OpenAI开发的模型，GPT-4有100万亿参数。",  # Wrong
        }
        print("✓ Pure Generator initialized (no retrieval)")
    
    def generate(self, question):
        """Generate answer without retrieval (may hallucinate)"""
        question_lower = question.lower()
        
        for key, answer in self.knowledge.items():
            if key in question_lower:
                return {
                    "answer": answer,
                    "source": "model_memory",
                    "grounded": False,
                    "confidence": "low"
                }
        
        return {
            "answer": "我不确定，但我猜测这可能与深度学习有关...",
            "source": "model_memory",
            "grounded": False,
            "confidence": "very_low"
        }


class RAGGenerator:
    """Simulates RAG - retrieves then generates"""
    
    def __init__(self, knowledge_base):
        self.knowledge_base = knowledge_base
        print(f"✓ RAG Generator initialized ({len(knowledge_base)} documents)")
    
    def retrieve(self, question, top_k=2):
        """Simple keyword-based retrieval"""
        scores = []
        q_chars = set(question.lower())
        
        for doc in self.knowledge_base:
            d_chars = set(doc["text"].lower())
            overlap = len(q_chars & d_chars) / max(len(q_chars), 1)
            scores.append((doc, overlap))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        return [s[0] for s in scores[:top_k]]
    
    def generate(self, question):
        """Generate answer with retrieval augmentation"""
        retrieved = self.retrieve(question)
        
        # Combine retrieved context
        context = " ".join([doc["text"] for doc in retrieved])
        
        # Generate grounded answer (simulated)
        answer = f"根据检索到的资料：{retrieved[0]['text']}"
        
        return {
            "answer": answer,
            "source": "retrieved_documents",
            "grounded": True,
            "confidence": "high",
            "citations": [doc["title"] for doc in retrieved],
            "num_sources": len(retrieved)
        }


# === Demo ===
print("=" * 60)
print("RAG vs Pure Generation Comparison")
print("=" * 60)

# Ground truth knowledge base
knowledge_base = [
    {"title": "Transformer论文", 
     "text": "Transformer由Vaswani等人在2017年提出，是基于自注意力机制的序列到序列模型，最初用于机器翻译任务。"},
    {"title": "BERT技术报告",
     "text": "BERT-base有1.1亿参数，BERT-large有3.4亿参数。BERT使用掩码语言模型(MLM)和下一句预测(NSP)进行预训练。"},
    {"title": "GPT系列介绍",
     "text": "GPT-3有1750亿参���，GPT-4的具体参数量未公开。GPT系列使用自回归方式预测下一个token。"},
]

pure_gen = PureGenerator()
rag_gen = RAGGenerator(knowledge_base)

questions = [
    "Transformer是什么？",
    "BERT有多少参数？",
    "GPT-3有多少参数？",
]

for q in questions:
    print(f"\n❓ Question: {q}")
    
    # Pure generation
    pure_result = pure_gen.generate(q)
    print(f"\n  🤖 Pure Generation:")
    print(f"     Answer: {pure_result['answer']}")
    print(f"     Grounded: {pure_result['grounded']}, Confidence: {pure_result['confidence']}")
    
    # RAG generation
    rag_result = rag_gen.generate(q)
    print(f"\n  📚 RAG Generation:")
    print(f"     Answer: {rag_result['answer']}")
    print(f"     Grounded: {rag_result['grounded']}, Confidence: {rag_result['confidence']}")
    print(f"     Sources: {rag_result['citations']}")

print(f"\n{'='*60}")
print("💡 Key Takeaway:")
print("  Pure generation may hallucinate facts (wrong numbers, dates)")
print("  RAG grounds answers in retrieved documents → more reliable")
print("  RAG enables source attribution → verifiable answers")
print("=" * 60)

print("\n✅ RAG vs Pure Generation comparison complete!")

## 4. 嵌入模型与向量检索

### 4.1 文本嵌入

将文本映射到稠密向量空间，语义相似的文本在向量空间中距离更近：

$$\text{embedding}: \text{text} \rightarrow \mathbb{R}^d$$

$$\text{similarity}(q, d) = \cos(\theta) = \frac{\mathbf{q} \cdot \mathbf{d}}{\|\mathbf{q}\| \|\mathbf{d}\|}$$

### 4.2 嵌入模型对比

| 模型 | 维度 | 语言 | 特点 |
|------|------|------|------|
| **all-MiniLM-L6-v2** | 384 | 英文 | 轻量快速，适合原型 |
| **BGE-large-zh** | 1024 | 中文 | 中文效果最佳 |
| **E5-large-v2** | 1024 | 英文 | 高质量通用嵌入 |
| **GTE-large** | 1024 | 多语言 | 阿里开源，效果好 |
| **text-embedding-3-large** | 3072 | 多语言 | OpenAI API，效果顶尖 |

### 4.3 向量索引类型

| 索引类型 | 搜索方式 | 速度 | 精度 | 内存 |
|----------|---------|------|------|------|
| **Flat (暴力搜索)** | 精确 | 慢 | 100% | 高 |
| **IVF (倒排索引)** | 近似 | 快 | 95-99% | 中 |
| **HNSW (图索引)** | 近似 | 很快 | 97-99% | 高 |
| **PQ (乘积量化)** | 近似 | 快 | 90-95% | 低 |

> **推荐：** 小规模用Flat，中规模用HNSW，大规模用IVF+PQ

In [ ]:
# 🔬 Micro Practice: Embedding Models and Vector Search
# Goal: Implement text embeddings and FAISS-style vector search
# Expected outcome: Understand embedding generation, similarity search, and indexing

class TextEmbedder:
    """Text embedding model (simulated Sentence-BERT style)"""
    
    def __init__(self, embedding_dim=128):
        self.embedding_dim = embedding_dim
        self.vocab = {}
        self.vocab_embeddings = {}
        print(f"✓ Text Embedder initialized (dim={embedding_dim})")
    
    def _char_embedding(self, char):
        """Generate deterministic embedding for a character"""
        np.random.seed(ord(char) % 2**31)
        return np.random.randn(self.embedding_dim).astype(np.float32)
    
    def encode(self, text):
        """Encode text to dense vector (simulated)"""
        if not text:
            return np.zeros(self.embedding_dim, dtype=np.float32)
        
        # Average character embeddings (simplified sentence embedding)
        char_embs = [self._char_embedding(c) for c in text if c.strip()]
        if not char_embs:
            return np.zeros(self.embedding_dim, dtype=np.float32)
        
        embedding = np.mean(char_embs, axis=0)
        # L2 normalize
        norm = np.linalg.norm(embedding)
        if norm > 0:
            embedding = embedding / norm
        return embedding
    
    def encode_batch(self, texts):
        """Encode multiple texts"""
        return np.array([self.encode(text) for text in texts])
    
    def similarity(self, text1, text2):
        """Compute cosine similarity between two texts"""
        emb1 = self.encode(text1)
        emb2 = self.encode(text2)
        return float(np.dot(emb1, emb2))


class VectorIndex:
    """FAISS-style vector index for similarity search"""
    
    def __init__(self, embedding_dim):
        self.embedding_dim = embedding_dim
        self.vectors = None
        self.documents = []
        self.n_vectors = 0
        print(f"✓ Vector Index initialized (dim={embedding_dim})")
    
    def add(self, vectors, documents):
        """Add vectors and associated documents to index"""
        vectors = np.array(vectors, dtype=np.float32)
        
        if self.vectors is None:
            self.vectors = vectors
        else:
            self.vectors = np.vstack([self.vectors, vectors])
        
        self.documents.extend(documents)
        self.n_vectors = len(self.documents)
        print(f"  Added {len(documents)} vectors (total: {self.n_vectors})")
    
    def search(self, query_vector, k=5):
        """Find k nearest neighbors using cosine similarity"""
        query_vector = np.array(query_vector, dtype=np.float32)
        
        # Cosine similarity (vectors are normalized)
        similarities = self.vectors @ query_vector
        
        # Get top-k indices
        top_k_indices = np.argsort(similarities)[::-1][:k]
        
        results = []
        for idx in top_k_indices:
            results.append({
                "document": self.documents[idx],
                "score": float(similarities[idx]),
                "index": int(idx)
            })
        
        return results
    
    def search_batch(self, query_vectors, k=5):
        """Batch search for multiple queries"""
        return [self.search(qv, k) for qv in query_vectors]
    
    def stats(self):
        """Print index statistics"""
        print(f"\n📊 Index Statistics:")
        print(f"  Vectors: {self.n_vectors}")
        print(f"  Dimension: {self.embedding_dim}")
        if self.vectors is not None:
            print(f"  Memory: {self.vectors.nbytes / 1024:.1f} KB")
            norms = np.linalg.norm(self.vectors, axis=1)
            print(f"  Norm range: [{norms.min():.4f}, {norms.max():.4f}]")


# === Demo: Text Embeddings ===
print("=" * 60)
print("Part 1: Text Embeddings")
print("=" * 60)

embedder = TextEmbedder(embedding_dim=128)

# Demonstrate semantic similarity
pairs = [
    ("Transformer是一种神经网络架构", "Transformer是深度学习模型"),
    ("Transformer是一种神经网络架构", "今天天气很好"),
    ("注意力机制计算查询和键的相似度", "自注意力通过点积计算权重"),
    ("注意力机制计算查询和键的相似度", "猫喜欢吃鱼"),
]

print("\n📐 Cosine Similarity:")
for text1, text2 in pairs:
    sim = embedder.similarity(text1, text2)
    bar = "█" * int(sim * 20) if sim > 0 else ""
    print(f"  {sim:+.4f} {bar}")
    print(f"    A: {text1}")
    print(f"    B: {text2}")

# === Part 2: Vector Index ===
print(f"\n{'='*60}")
print("Part 2: Vector Search")
print("=" * 60)

# Build knowledge base
documents = [
    "Transformer是一种基于自注意力机制的神经网络架构",
    "BERT使用掩码语言模型进行预训练",
    "GPT是自回归语言模型，预测下一个token",
    "注意力机制通过查询键值计算权重",
    "词嵌入将词语映射到连续向量空间",
    "微调是将预训练模型适配到下游任务",
    "量化通过降低精度来压缩模型",
    "知识蒸馏让小模型学习大模型的知识",
    "LoRA通过低秩矩阵实现高效微调",
    "ONNX是跨框架的模型交换格式",
]

# Create embeddings and index
embeddings = embedder.encode_batch(documents)
index = VectorIndex(embedding_dim=128)
index.add(embeddings, documents)
index.stats()

# Search
queries = [
    "什么是Transformer？",
    "如何压缩模型？",
    "预训练方法有哪些？",
]

print(f"\n🔍 Search Results:")
for query in queries:
    query_emb = embedder.encode(query)
    results = index.search(query_emb, k=3)
    
    print(f"\n  Query: {query}")
    for i, r in enumerate(results):
        print(f"    Top-{i+1} (score={r['score']:.4f}): {r['document']}")

# Benchmark search speed
import time
n_queries = 100
query_embs = [embedder.encode(f"测试查询{i}") for i in range(n_queries)]

start = time.time()
for qe in query_embs:
    index.search(qe, k=5)
elapsed = time.time() - start

print(f"\n⚡ Search Benchmark:")
print(f"  {n_queries} queries in {elapsed*1000:.1f}ms")
print(f"  {elapsed/n_queries*1000:.2f}ms per query")
print(f"  {n_queries/elapsed:.0f} queries/second")

print("\n✅ Embeddings and vector search complete!")

## 5. 文档处理与分块

### 5.1 分块策略

| 策略 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| **固定大小** | 按字符/token数切分 | 简单、均匀 | 可能切断语义 |
| **句子分块** | 按句子边界切分 | 保持语义完整 | 大小不均匀 |
| **语义分块** | 按主题/段落切分 | 语义连贯 | 实现复杂 |
| **重叠分块** | 固定大小+重叠区域 | 保持上下文连续 | 增加存储 |

### 5.2 分块参数选择

```
chunk_size (分块大小):
  - 太小 (<100 tokens): 缺少上下文，检索噪声大
  - 适中 (256-512 tokens): 平衡信息量和精度
  - 太大 (>1000 tokens): 包含无关信息，降低精度

overlap (重叠比例):
  - 0%: 无重叠，可能丢失边界信息
  - 10-20%: 推荐，保持上下文连续性
  - >30%: 冗余过多，增加存储和计算成本
```

### 5.3 元数据标注

为每个文档块添加元数据，支持过滤检索：

| 元数据 | 用途 | 示例 |
|--------|------|------|
| **来源** | 溯源 | "paper_2024.pdf" |
| **类别** | 过滤 | "architecture", "training" |
| **日期** | 时效性 | "2024-01-15" |
| **章节** | 层次化检索 | "Chapter 3.2" |
| **语言** | 多语言支持 | "zh", "en" |

In [ ]:
# 🔬 Micro Practice: Document Processing and Chunking
# Goal: Implement multiple chunking strategies and compare quality
# Expected outcome: Understand how chunking affects retrieval quality

class DocumentChunker:
    """Multiple chunking strategies for RAG"""
    
    def __init__(self):
        print("✓ Document Chunker initialized")
    
    def fixed_size_chunks(self, text, chunk_size=100, overlap=0):
        """Split text into fixed-size character chunks"""
        chunks = []
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk = text[start:end]
            if chunk.strip():
                chunks.append({
                    "text": chunk.strip(),
                    "start": start,
                    "end": min(end, len(text)),
                    "method": "fixed_size"
                })
            start += chunk_size - overlap
        return chunks
    
    def sentence_chunks(self, text, max_sentences=3):
        """Split text into sentence-based chunks"""
        # Split on sentence boundaries
        sentences = []
        for sep in ["。", "！", "？", ". ", "! ", "? "]:
            if sep in text:
                parts = text.split(sep)
                sentences = [p.strip() + sep.strip() for p in parts if p.strip()]
                break
        
        if not sentences:
            sentences = [text]
        
        # Group sentences
        chunks = []
        for i in range(0, len(sentences), max_sentences):
            group = sentences[i:i + max_sentences]
            chunk_text = "".join(group)
            if chunk_text.strip():
                chunks.append({
                    "text": chunk_text.strip(),
                    "start_sentence": i,
                    "end_sentence": min(i + max_sentences, len(sentences)),
                    "method": "sentence"
                })
        return chunks
    
    def semantic_chunks(self, text, similarity_threshold=0.3):
        """Split text by semantic boundaries (simplified)"""
        # Split into paragraphs first
        paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
        
        if len(paragraphs) <= 1:
            # Fall back to sentence splitting
            paragraphs = [s.strip() for s in text.split("。") if s.strip()]
        
        # Group similar paragraphs
        chunks = []
        current_chunk = []
        
        for para in paragraphs:
            if not current_chunk:
                current_chunk.append(para)
            else:
                # Simple similarity: shared character ratio
                prev_chars = set(current_chunk[-1])
                curr_chars = set(para)
                similarity = len(prev_chars & curr_chars) / max(len(prev_chars | curr_chars), 1)
                
                if similarity >= similarity_threshold:
                    current_chunk.append(para)
                else:
                    chunks.append({
                        "text": "。".join(current_chunk),
                        "num_paragraphs": len(current_chunk),
                        "method": "semantic"
                    })
                    current_chunk = [para]
        
        if current_chunk:
            chunks.append({
                "text": "。".join(current_chunk),
                "num_paragraphs": len(current_chunk),
                "method": "semantic"
            })
        
        return chunks
    
    def overlapping_chunks(self, text, chunk_size=100, overlap_ratio=0.2):
        """Fixed-size chunks with overlap"""
        overlap = int(chunk_size * overlap_ratio)
        return self.fixed_size_chunks(text, chunk_size, overlap)


class MetadataEnricher:
    """Add metadata to document chunks for filtered retrieval"""
    
    def __init__(self):
        self.category_keywords = {
            "architecture": ["transformer", "架构", "编码器", "解码器", "模型结构"],
            "training": ["训练", "预训练", "微调", "学习率", "优化"],
            "nlp": ["语言", "文本", "词", "句子", "语义"],
            "deployment": ["部署", "推理", "量化", "压缩", "优化"],
        }
        print("✓ Metadata Enricher initialized")
    
    def enrich(self, chunk, source_doc=None):
        """Add metadata to a chunk"""
        text_lower = chunk["text"].lower()
        
        # Auto-detect category
        categories = []
        for cat, keywords in self.category_keywords.items():
            if any(kw in text_lower for kw in keywords):
                categories.append(cat)
        
        chunk["metadata"] = {
            "categories": categories or ["general"],
            "char_count": len(chunk["text"]),
            "word_count": len(chunk["text"].split()),
            "source": source_doc or "unknown",
            "has_code": "```" in chunk["text"] or "def " in chunk["text"],
        }
        
        return chunk
    
    def filtered_search(self, chunks, query, category_filter=None):
        """Search chunks with metadata filtering"""
        filtered = chunks
        
        if category_filter:
            filtered = [c for c in chunks 
                       if any(cat in c.get("metadata", {}).get("categories", []) 
                             for cat in category_filter)]
        
        # Simple relevance scoring
        results = []
        query_chars = set(query)
        for chunk in filtered:
            chunk_chars = set(chunk["text"])
            score = len(query_chars & chunk_chars) / max(len(query_chars), 1)
            results.append({**chunk, "score": score})
        
        results.sort(key=lambda x: x["score"], reverse=True)
        return results


# === Demo ===
print("=" * 60)
print("Document Chunking Strategies")
print("=" * 60)

# Sample document
document = (
    "Transformer是一种基于自注意力机制的神经网络架构。"
    "它由编码器和解码器组成，通过多头注意力机制捕获序列中的长距离依赖关系。"
    "Transformer摒弃了传统的循环结构，完全依赖注意力机制进行序列建模。"
    "BERT是基于Transformer编码器的预训练模型。"
    "它使用掩码语言模型和下一句预测两个任务进行预训练。"
    "BERT是双向的，能同时利用左右上下文信息。"
    "GPT是基于Transformer解码器的自回归语言模型。"
    "GPT通过预测下一个token来进行预训练。"
    "GPT-3拥有1750亿参数，展示了强大的少样本学习能力。"
)

chunker = DocumentChunker()

# Compare strategies
strategies = {
    "Fixed (100 chars)": chunker.fixed_size_chunks(document, chunk_size=100),
    "Sentence (2 per chunk)": chunker.sentence_chunks(document, max_sentences=2),
    "Overlapping (100, 20%)": chunker.overlapping_chunks(document, chunk_size=100, overlap_ratio=0.2),
    "Semantic": chunker.semantic_chunks(document, similarity_threshold=0.3),
}

for name, chunks in strategies.items():
    print(f"\n📋 {name}: {len(chunks)} chunks")
    for i, chunk in enumerate(chunks):
        preview = chunk["text"][:60] + "..." if len(chunk["text"]) > 60 else chunk["text"]
        print(f"  Chunk {i}: [{len(chunk['text'])} chars] {preview}")

# Metadata enrichment
print(f"\n{'='*60}")
print("Metadata Enrichment")
print("=" * 60)

enricher = MetadataEnricher()
sentence_chunks = chunker.sentence_chunks(document, max_sentences=3)

for chunk in sentence_chunks:
    enricher.enrich(chunk, source_doc="transformer_guide.md")

for i, chunk in enumerate(sentence_chunks):
    meta = chunk["metadata"]
    print(f"\nChunk {i}: {chunk['text'][:50]}...")
    print(f"  Categories: {meta['categories']}")
    print(f"  Words: {meta['word_count']}, Source: {meta['source']}")

# Filtered search
print(f"\n--- Filtered Search ---")
results = enricher.filtered_search(sentence_chunks, "BERT预训练", category_filter=["training"])
print(f"Query: 'BERT预训练' (filter: training)")
for r in results[:2]:
    print(f"  Score: {r['score']:.2f} | {r['text'][:60]}...")

print("\n✅ Document processing and chunking complete!")

## 6. 检索策略

### 6.1 检索方法对比

| 方法 | 原理 | 优势 | 劣势 |
|------|------|------|------|
| **稠密检索** | 语义向量相似度 | 理解语义、泛化好 | 精确匹配弱 |
| **稀疏检索 (BM25)** | 词频统计匹配 | 精确匹配强、可解释 | 无法理解语义 |
| **混合检索** | 稠密 + 稀疏融合 | 兼顾语义和精确匹配 | 需要调权重 |
| **重排序** | 交叉编码器精排 | 精度最高 | 计算成本高 |

### 6.2 BM25公式

$$\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t, d) \cdot (k_1 + 1)}{f(t, d) + k_1 \cdot (1 - b + b \cdot \frac{|d|}{\text{avgdl}})}$$

- $f(t, d)$: 词项t在文档d中的频率
- $|d|$: 文档长度
- $\text{avgdl}$: 平均文档长度
- $k_1$: 词频饱和参数（通常1.2-2.0）
- $b$: 长度归一化参数（通常0.75）

### 6.3 混合检索融合

```
最终分数 = α × dense_score + (1-α) × sparse_score

α的选择:
  α = 0.3: 偏向精确匹配（技术文档、代码搜索）
  α = 0.5: 均衡（通用问答）
  α = 0.7: 偏向语义理解（开放域问答）
```

### 6.4 两阶段检索

```
Stage 1: 粗排 (Retrieval)
  - BM25 或 Dense Retrieval
  - 从百万文档中召回 Top-100
  - 速度优先

Stage 2: 精排 (Re-ranking)
  - Cross-Encoder 逐对打分
  - 从 Top-100 中选出 Top-5
  - 精度优先
```

In [ ]:
# 🔬 Micro Practice: Advanced Retrieval Strategies
# Goal: Implement hybrid retrieval and re-ranking
# Expected outcome: Compare dense, sparse, and hybrid retrieval methods

import math
from collections import Counter

class BM25Retriever:
    """BM25 sparse retrieval"""
    
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.documents = []
        self.doc_lengths = []
        self.avg_dl = 0
        self.df = {}  # Document frequency
        self.tf = []  # Term frequency per document
        self.N = 0
        print(f"✓ BM25 Retriever initialized (k1={k1}, b={b})")
    
    def _tokenize(self, text):
        """Simple tokenization"""
        # Split on spaces and Chinese characters
        tokens = []
        current = []
        for char in text.lower():
            if '\u4e00' <= char <= '\u9fff':  # Chinese character
                if current:
                    tokens.append(''.join(current))
                    current = []
                tokens.append(char)
            elif char.isalnum():
                current.append(char)
            else:
                if current:
                    tokens.append(''.join(current))
                    current = []
        if current:
            tokens.append(''.join(current))
        return tokens
    
    def index(self, documents):
        """Build BM25 index"""
        self.documents = documents
        self.N = len(documents)
        self.tf = []
        self.df = {}
        self.doc_lengths = []
        
        for doc in documents:
            tokens = self._tokenize(doc["text"])
            self.doc_lengths.append(len(tokens))
            
            # Term frequency
            tf = Counter(tokens)
            self.tf.append(tf)
            
            # Document frequency
            for term in set(tokens):
                self.df[term] = self.df.get(term, 0) + 1
        
        self.avg_dl = sum(self.doc_lengths) / self.N if self.N > 0 else 0
        print(f"  Indexed {self.N} documents, {len(self.df)} unique terms")
    
    def _score(self, query_tokens, doc_idx):
        """Calculate BM25 score for a document"""
        score = 0.0
        dl = self.doc_lengths[doc_idx]
        
        for term in query_tokens:
            if term not in self.df:
                continue
            
            # IDF
            idf = math.log((self.N - self.df[term] + 0.5) / (self.df[term] + 0.5) + 1)
            
            # TF with length normalization
            tf = self.tf[doc_idx].get(term, 0)
            tf_norm = (tf * (self.k1 + 1)) / (tf + self.k1 * (1 - self.b + self.b * dl / self.avg_dl))
            
            score += idf * tf_norm
        
        return score
    
    def search(self, query, k=5):
        """Search documents"""
        query_tokens = self._tokenize(query)
        
        scores = []
        for i in range(self.N):
            score = self._score(query_tokens, i)
            scores.append((i, score))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        
        results = []
        for idx, score in scores[:k]:
            results.append({
                **self.documents[idx],
                "score": score,
                "method": "bm25"
            })
        return results


class DenseRetriever:
    """Dense retrieval using embeddings"""
    
    def __init__(self, embedding_dim=64):
        self.embedding_dim = embedding_dim
        self.documents = []
        self.embeddings = None
        print(f"✓ Dense Retriever initialized (dim={embedding_dim})")
    
    def _embed(self, text):
        """Generate embedding (simulated with hash-based vectors)"""
        np.random.seed(hash(text) % 2**31)
        emb = np.random.randn(self.embedding_dim).astype(np.float32)
        return emb / np.linalg.norm(emb)
    
    def index(self, documents):
        """Build dense index"""
        self.documents = documents
        self.embeddings = np.array([self._embed(doc["text"]) for doc in documents])
        print(f"  Indexed {len(documents)} documents, embedding shape: {self.embeddings.shape}")
    
    def search(self, query, k=5):
        """Search by cosine similarity"""
        query_emb = self._embed(query)
        
        # Cosine similarity (embeddings are normalized)
        similarities = self.embeddings @ query_emb
        
        top_indices = np.argsort(similarities)[::-1][:k]
        
        results = []
        for idx in top_indices:
            results.append({
                **self.documents[idx],
                "score": float(similarities[idx]),
                "method": "dense"
            })
        return results


class HybridRetriever:
    """Hybrid retrieval combining dense and sparse methods"""
    
    def __init__(self, dense_weight=0.5, sparse_weight=0.5):
        self.dense = DenseRetriever()
        self.sparse = BM25Retriever()
        self.dense_weight = dense_weight
        self.sparse_weight = sparse_weight
        print(f"✓ Hybrid Retriever initialized (dense={dense_weight}, sparse={sparse_weight})")
    
    def index(self, documents):
        """Index documents in both retrievers"""
        self.dense.index(documents)
        self.sparse.index(documents)
    
    def _normalize_scores(self, results):
        """Min-max normalize scores"""
        if not results:
            return results
        scores = [r["score"] for r in results]
        min_s, max_s = min(scores), max(scores)
        range_s = max_s - min_s if max_s != min_s else 1.0
        
        for r in results:
            r["norm_score"] = (r["score"] - min_s) / range_s
        return results
    
    def search(self, query, k=5):
        """Hybrid search with score fusion"""
        # Get results from both methods
        dense_results = self._normalize_scores(self.dense.search(query, k=k*2))
        sparse_results = self._normalize_scores(self.sparse.search(query, k=k*2))
        
        # Merge scores by document ID
        score_map = {}
        
        for r in dense_results:
            doc_id = r["id"]
            score_map[doc_id] = {
                "doc": r,
                "dense_score": r["norm_score"],
                "sparse_score": 0.0
            }
        
        for r in sparse_results:
            doc_id = r["id"]
            if doc_id in score_map:
                score_map[doc_id]["sparse_score"] = r["norm_score"]
            else:
                score_map[doc_id] = {
                    "doc": r,
                    "dense_score": 0.0,
                    "sparse_score": r["norm_score"]
                }
        
        # Calculate hybrid scores
        results = []
        for doc_id, info in score_map.items():
            hybrid_score = (self.dense_weight * info["dense_score"] + 
                          self.sparse_weight * info["sparse_score"])
            results.append({
                **info["doc"],
                "score": hybrid_score,
                "dense_score": info["dense_score"],
                "sparse_score": info["sparse_score"],
                "method": "hybrid"
            })
        
        results.sort(key=lambda x: x["score"], reverse=True)
        return results[:k]


class CrossEncoderReranker:
    """Cross-encoder re-ranking for precision improvement"""
    
    def __init__(self):
        print("✓ Cross-Encoder Reranker initialized")
    
    def _score_pair(self, query, document):
        """Score query-document pair (simulated cross-encoder)"""
        # Simulate: higher score for more keyword overlap
        q_tokens = set(query.lower().split())
        d_tokens = set(document.lower().split())
        stop_words = {"的", "是", "在", "了", "和", "有", "什么", "如何"}
        q_tokens -= stop_words
        d_tokens -= stop_words
        
        if not q_tokens:
            return 0.5
        
        overlap = len(q_tokens & d_tokens)
        base_score = overlap / len(q_tokens)
        
        # Add some noise for realism
        np.random.seed(hash(query + document) % 2**31)
        noise = np.random.uniform(-0.1, 0.1)
        
        return min(max(base_score + noise, 0.0), 1.0)
    
    def rerank(self, query, results, top_k=None):
        """Re-rank results using cross-encoder scores"""
        reranked = []
        for r in results:
            ce_score = self._score_pair(query, r.get("text", ""))
            reranked.append({
                **r,
                "original_score": r["score"],
                "rerank_score": ce_score,
                "score": ce_score  # Replace with rerank score
            })
        
        reranked.sort(key=lambda x: x["rerank_score"], reverse=True)
        
        if top_k:
            reranked = reranked[:top_k]
        
        return reranked


# === Demo ===
print("=" * 60)
print("Retrieval Strategies Comparison")
print("=" * 60)

# Knowledge base
documents = [
    {"id": 1, "title": "Transformer", "text": "Transformer是基于自注意力机制的神经网络架构，用于序列建模任务。"},
    {"id": 2, "title": "注意力机制", "text": "注意力机制通过计算查询和键的相似度来分配权重，关注重要信息。"},
    {"id": 3, "title": "BERT", "text": "BERT使用掩码语言模型进行预训练，是双向的Transformer编码器。"},
    {"id": 4, "title": "GPT", "text": "GPT是自回归语言模型，通过预测下一个token进行预训练。"},
    {"id": 5, "title": "词嵌入", "text": "词嵌入将词语映射到向量空间，语义相似的词距离更近。"},
    {"id": 6, "title": "微调", "text": "微调将预训练模型适配到下游任务，LoRA是高效微调方法。"},
]

query = "Transformer的注意力机制是什么？"

# 1. BM25 (Sparse)
print(f"\n📝 Query: {query}")
print(f"\n--- BM25 (Sparse) ---")
bm25 = BM25Retriever()
bm25.index(documents)
bm25_results = bm25.search(query, k=3)
for r in bm25_results:
    print(f"  [{r['id']}] {r['title']}: score={r['score']:.4f}")

# 2. Dense
print(f"\n--- Dense Retrieval ---")
dense = DenseRetriever()
dense.index(documents)
dense_results = dense.search(query, k=3)
for r in dense_results:
    print(f"  [{r['id']}] {r['title']}: score={r['score']:.4f}")

# 3. Hybrid
print(f"\n--- Hybrid (Dense + Sparse) ---")
hybrid = HybridRetriever(dense_weight=0.4, sparse_weight=0.6)
hybrid.index(documents)
hybrid_results = hybrid.search(query, k=3)
for r in hybrid_results:
    print(f"  [{r['id']}] {r['title']}: hybrid={r['score']:.4f} "
          f"(dense={r.get('dense_score', 0):.2f}, sparse={r.get('sparse_score', 0):.2f})")

# 4. Re-ranking
print(f"\n--- With Cross-Encoder Re-ranking ---")
reranker = CrossEncoderReranker()
reranked = reranker.rerank(query, hybrid_results, top_k=3)
for r in reranked:
    print(f"  [{r['id']}] {r['title']}: rerank={r['rerank_score']:.4f} "
          f"(original={r['original_score']:.4f})")

print("\n✅ Retrieval strategies comparison complete!")

## 7. RAG流水线与生成

### 7.1 RAG流水线架构

```
用户查询 (Query)
    ↓
查询处理 (Query Processing)
    ↓ 查询改写、扩展
向量检索 (Retrieval)
    ↓ Top-K文档
上下文压缩 (Context Compression)
    ↓ 去除冗余、保留关键信息
提示构建 (Prompt Augmentation)
    ↓ 组合上下文 + 问题 + 指令
生成回答 (Generation)
    ↓ LLM生成
引用标注 (Citation)
    ↓ 标注来源
返回结果
```

### 7.2 Prompt工程

```python
# RAG Prompt模板
prompt = """
请基于以下参考资料回答问题。
规则：
1. 仅使用提供的资料回答
2. 如果资料不足，请说明
3. 引用来源编号 [1], [2] 等

参考资料:
{context}

问题: {question}

回答:
"""
```

### 7.3 上下文优化

| 技术 | 说明 | 效果 |
|------|------|------|
| **上下文压缩** | 去除无关句子 | 减少噪声，降低成本 |
| **相关性过滤** | 低分文档不纳入 | 提高答案质量 |
| **去重** | 合并重复信息 | 减少冗余 |
| **排序** | 最相关的放前面 | 利用LLM的位置偏好 |
| **长度控制** | 限制总token数 | 适配上下文窗口 |

In [ ]:
# 🔬 Micro Practice: RAG Pipeline and Generation
# Goal: Build end-to-end RAG pipeline with context optimization
# Expected outcome: Complete query → retrieve → augment → generate → cite pipeline

class ContextCompressor:
    """Compress retrieved context to fit within token limits"""
    
    def __init__(self, max_tokens=512):
        self.max_tokens = max_tokens
        print(f"✓ Context Compressor initialized (max_tokens={max_tokens})")
    
    def relevance_score(self, chunk, query):
        """Score chunk relevance to query (simplified)"""
        query_tokens = set(query.lower().split())
        chunk_tokens = set(chunk.lower().split())
        
        # Remove stop words
        stop_words = {"的", "是", "在", "了", "和", "有", "the", "a", "is", "in", "to", "for"}
        query_tokens -= stop_words
        chunk_tokens -= stop_words
        
        if not query_tokens:
            return 0.0
        
        overlap = len(query_tokens & chunk_tokens)
        return overlap / len(query_tokens)
    
    def compress(self, chunks, query, scores=None):
        """Compress chunks to fit within token limit"""
        # Score and rank chunks
        scored_chunks = []
        for i, chunk in enumerate(chunks):
            rel_score = scores[i] if scores else self.relevance_score(chunk, query)
            scored_chunks.append((chunk, rel_score, i))
        
        # Sort by relevance (highest first)
        scored_chunks.sort(key=lambda x: x[1], reverse=True)
        
        # Select chunks within token budget
        selected = []
        total_tokens = 0
        
        for chunk, score, orig_idx in scored_chunks:
            chunk_tokens = len(chunk.split())
            if total_tokens + chunk_tokens <= self.max_tokens:
                selected.append((chunk, score, orig_idx))
                total_tokens += chunk_tokens
        
        # Re-sort by original order for coherence
        selected.sort(key=lambda x: x[2])
        
        return {
            "compressed_chunks": [s[0] for s in selected],
            "scores": [s[1] for s in selected],
            "total_tokens": total_tokens,
            "chunks_selected": len(selected),
            "chunks_total": len(chunks),
            "compression_ratio": total_tokens / sum(len(c.split()) for c in chunks) if chunks else 0
        }


class RAGPipeline:
    """Complete RAG pipeline: Query → Retrieve → Augment → Generate → Cite"""
    
    def __init__(self, knowledge_base, compressor=None, top_k=3):
        self.knowledge_base = knowledge_base
        self.compressor = compressor or ContextCompressor()
        self.top_k = top_k
        self.history = []
        print(f"✓ RAG Pipeline initialized (top_k={top_k})")
    
    def retrieve(self, query):
        """Step 1: Retrieve relevant documents"""
        results = []
        query_tokens = set(query.lower().split())
        
        for doc in self.knowledge_base:
            doc_tokens = set(doc["text"].lower().split())
            # Simple relevance scoring
            stop_words = {"的", "是", "在", "了", "和", "有"}
            overlap = len((query_tokens - stop_words) & (doc_tokens - stop_words))
            score = overlap / max(len(query_tokens - stop_words), 1)
            results.append({**doc, "score": score})
        
        # Sort by score and return top-k
        results.sort(key=lambda x: x["score"], reverse=True)
        return results[:self.top_k]
    
    def augment(self, query, retrieved_docs):
        """Step 2: Build augmented prompt with context"""
        # Compress context
        chunks = [doc["text"] for doc in retrieved_docs]
        scores = [doc["score"] for doc in retrieved_docs]
        compressed = self.compressor.compress(chunks, query, scores)
        
        # Build prompt
        context = "\n\n".join(f"[来源{i+1}] {chunk}" 
                              for i, chunk in enumerate(compressed["compressed_chunks"]))
        
        prompt = f"""请基于以下参考资料回答问题。如果资料中没有相关信息，请说明。

参考资料:
{context}

问题: {query}

回答（请引用来源编号）:"""
        
        return prompt, compressed
    
    def generate(self, prompt, retrieved_docs):
        """Step 3: Generate answer (simulated)"""
        # In production, this calls an LLM
        # Here we simulate by extracting key sentences from retrieved docs
        
        answer_parts = []
        for i, doc in enumerate(retrieved_docs):
            # Take first sentence as key info
            sentences = doc["text"].split("。")
            if sentences:
                answer_parts.append(f"{sentences[0]}（[来源{i+1}]）")
        
        answer = "。".join(answer_parts[:3]) + "。"
        
        citations = [{"source_id": i+1, "title": doc.get("title", f"Document {i+1}"),
                       "score": doc["score"]}
                      for i, doc in enumerate(retrieved_docs)]
        
        return answer, citations
    
    def query(self, question):
        """Full RAG pipeline"""
        # Step 1: Retrieve
        retrieved = self.retrieve(question)
        
        # Step 2: Augment
        prompt, compression_info = self.augment(question, retrieved)
        
        # Step 3: Generate
        answer, citations = self.generate(prompt, retrieved)
        
        result = {
            "question": question,
            "answer": answer,
            "citations": citations,
            "retrieved_count": len(retrieved),
            "compression": compression_info,
            "prompt_preview": prompt[:200] + "..."
        }
        
        self.history.append(result)
        return result
    
    def print_result(self, result):
        """Pretty print RAG result"""
        print(f"\n{'='*60}")
        print(f"❓ Question: {result['question']}")
        print(f"{'='*60}")
        print(f"\n📝 Answer:\n{result['answer']}")
        print(f"\n📚 Citations:")
        for cite in result['citations']:
            print(f"  [{cite['source_id']}] {cite['title']} (relevance: {cite['score']:.2f})")
        print(f"\n📊 Stats:")
        print(f"  Retrieved: {result['retrieved_count']} documents")
        comp = result['compression']
        print(f"  Context: {comp['chunks_selected']}/{comp['chunks_total']} chunks, "
              f"{comp['total_tokens']} tokens")


# === Demo: Build Knowledge Base ===
print("=" * 60)
print("RAG Pipeline Demo")
print("=" * 60)

knowledge_base = [
    {"id": 1, "title": "Transformer架构",
     "text": "Transformer是一种基于自注意力机制的神经网络架构。它由编码器和解码器组成，通过多头注意力机制捕获序列中的长距离依赖关系。Transformer摒弃了传统的循环结构，完全依赖注意力机制进行序列建模。"},
    {"id": 2, "title": "注意力机制",
     "text": "注意力机制允许模型在处理每个位置时关注输入序列的不同部分。自注意力通过计算查询、键和值的点积来确定注意力权重。多头注意力将注意力分成多个头，每个头关注不同的表示子空间。"},
    {"id": 3, "title": "BERT预训练",
     "text": "BERT使用掩码语言模型和下一句预测两个任务进行预训练。掩码语言模型随机遮盖输入中15%的token，让模型预测被遮盖的内容。BERT是双向的，能同时利用左右上下文信息。"},
    {"id": 4, "title": "GPT系列",
     "text": "GPT是基于Transformer解码器的自回归语言模型。GPT通过预测下一个token来进行预训练。GPT-3拥有1750亿参数，展示了强大的少样本学习能力。GPT-4是多模态模型，支持文本和图像输入。"},
    {"id": 5, "title": "微调技术",
     "text": "微调是将预训练模型适配到下游任务的过程。常见方法包括全参数微调、LoRA低秩适配和Prefix Tuning。LoRA通过低秩矩阵分解大幅减少可训练参数数量，是目前最流行的高效微调方法。"},
    {"id": 6, "title": "词嵌入",
     "text": "词嵌入将离散的词语映射到连续的向量空间。Word2Vec通过Skip-gram或CBOW模型学习词向量。上下文相似的词在向量空间中距离更近。现代方法如BERT生成上下文相关的动态词嵌入。"},
    {"id": 7, "title": "模型量化",
     "text": "量化通过降低数值精度来压缩模型。常见方法包括INT8量化和INT4量化。动态量化在推理时进行，静态量化需要校准数据。量化可以将模型大小减少2-4倍，同时保持大部分精度。"},
    {"id": 8, "title": "知识蒸馏",
     "text": "知识蒸馏让小模型（学生）学习大模型（教师）的输出分布。通过软标签传递暗知识，温度参数控制知识传递的程度。DistilBERT通过蒸馏保留了BERT 97%的性能，同时减少了40%的参数。"},
]

# Create pipeline
pipeline = RAGPipeline(knowledge_base, ContextCompressor(max_tokens=200), top_k=3)

# Test queries
queries = [
    "什么是Transformer的注意力机制？",
    "BERT是如何预训练的？",
    "如何压缩和优化模型？",
]

for q in queries:
    result = pipeline.query(q)
    pipeline.print_result(result)

print("\n✅ RAG pipeline complete!")

## 8. RAG评估

### 8.1 评估维度

| 维度 | 指标 | 说明 |
|------|------|------|
| **检索质量** | Recall@K | Top-K中包含多少相关文档 |
| | Precision@K | Top-K中有多少是相关的 |
| | MRR | 第一个相关文档的排名倒数 |
| | NDCG@K | 考虑相关性等级的排名质量 |
| **生成质量** | ROUGE-L | 最长公共子序列匹配 |
| | BERTScore | 基于语义的相似度 |
| **端到端** | Faithfulness | 答案是否忠实于上下文 |
| | Answer Relevance | 答案是否回答了问题 |

### 8.2 NDCG计算

$$\text{DCG@K} = \sum_{i=1}^{K} \frac{2^{rel_i} - 1}{\log_2(i+1)}$$

$$\text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}$$

- IDCG是理想排序下的DCG
- NDCG ∈ [0, 1]，越高越好
- 考虑了文档的相关性等级（不仅是相关/不相关）

### 8.3 忠实度评估

```
忠实度 (Faithfulness):
  答案中的每个声明是否都能在上下文中找到支持？

答案相关性 (Answer Relevance):
  答案是否真正回答了用户的问题？

常见问题:
  ✗ 幻觉: 答案包含上下文中没有的信息
  ✗ 不完整: 答案遗漏了关键信息
  ✗ 偏题: 答案没有回答问题
```

In [ ]:
# 🔬 Micro Practice: RAG Evaluation Framework
# Goal: Implement comprehensive RAG evaluation metrics
# Expected outcome: Evaluate retrieval quality, generation quality, and end-to-end performance

class RetrievalEvaluator:
    """Evaluate retrieval quality"""
    
    def __init__(self):
        print("✓ Retrieval Evaluator initialized")
    
    def recall_at_k(self, retrieved_ids, relevant_ids, k):
        """Recall@K: fraction of relevant docs in top-K"""
        top_k = retrieved_ids[:k]
        relevant_found = len(set(top_k) & set(relevant_ids))
        return relevant_found / len(relevant_ids) if relevant_ids else 0.0
    
    def precision_at_k(self, retrieved_ids, relevant_ids, k):
        """Precision@K: fraction of top-K that are relevant"""
        top_k = retrieved_ids[:k]
        relevant_found = len(set(top_k) & set(relevant_ids))
        return relevant_found / k if k > 0 else 0.0
    
    def mrr(self, retrieved_ids, relevant_ids):
        """Mean Reciprocal Rank: 1/rank of first relevant doc"""
        for i, doc_id in enumerate(retrieved_ids):
            if doc_id in relevant_ids:
                return 1.0 / (i + 1)
        return 0.0
    
    def ndcg_at_k(self, retrieved_ids, relevance_scores, k):
        """NDCG@K: Normalized Discounted Cumulative Gain"""
        import math
        
        # DCG
        dcg = 0.0
        for i, doc_id in enumerate(retrieved_ids[:k]):
            rel = relevance_scores.get(doc_id, 0)
            dcg += (2**rel - 1) / math.log2(i + 2)
        
        # Ideal DCG
        ideal_rels = sorted(relevance_scores.values(), reverse=True)[:k]
        idcg = sum((2**rel - 1) / math.log2(i + 2) for i, rel in enumerate(ideal_rels))
        
        return dcg / idcg if idcg > 0 else 0.0
    
    def evaluate(self, retrieved_ids, relevant_ids, relevance_scores=None, k_values=[1, 3, 5, 10]):
        """Run all retrieval metrics"""
        results = {}
        for k in k_values:
            results[f"Recall@{k}"] = self.recall_at_k(retrieved_ids, relevant_ids, k)
            results[f"Precision@{k}"] = self.precision_at_k(retrieved_ids, relevant_ids, k)
        
        results["MRR"] = self.mrr(retrieved_ids, relevant_ids)
        
        if relevance_scores:
            for k in k_values:
                results[f"NDCG@{k}"] = self.ndcg_at_k(retrieved_ids, relevance_scores, k)
        
        return results


class GenerationEvaluator:
    """Evaluate generation quality"""
    
    def __init__(self):
        print("✓ Generation Evaluator initialized")
    
    def rouge_l(self, prediction, reference):
        """ROUGE-L: Longest Common Subsequence based metric"""
        pred_tokens = prediction.lower().split()
        ref_tokens = reference.lower().split()
        
        if not pred_tokens or not ref_tokens:
            return {"precision": 0, "recall": 0, "f1": 0}
        
        # LCS using dynamic programming
        m, n = len(pred_tokens), len(ref_tokens)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if pred_tokens[i-1] == ref_tokens[j-1]:
                    dp[i][j] = dp[i-1][j-1] + 1
                else:
                    dp[i][j] = max(dp[i-1][j], dp[i][j-1])
        
        lcs_length = dp[m][n]
        precision = lcs_length / m if m > 0 else 0
        recall = lcs_length / n if n > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        return {"precision": precision, "recall": recall, "f1": f1}
    
    def faithfulness(self, answer, context):
        """Faithfulness: check if answer claims are supported by context"""
        # Simplified: check token overlap between answer and context
        answer_tokens = set(answer.lower().split())
        context_tokens = set(context.lower().split())
        
        # Remove common stop words
        stop_words = {"the", "a", "an", "is", "are", "was", "were", "in", "on", "at",
                      "to", "for", "of", "with", "by", "的", "是", "在", "了", "和", "有"}
        answer_tokens -= stop_words
        context_tokens -= stop_words
        
        if not answer_tokens:
            return 1.0
        
        overlap = len(answer_tokens & context_tokens)
        return overlap / len(answer_tokens)
    
    def answer_relevance(self, answer, question):
        """Answer relevance: how well does the answer address the question"""
        # Simplified: check keyword overlap
        q_tokens = set(question.lower().split()) - {"什么", "如何", "为什么", "怎么", "?", "？"}
        a_tokens = set(answer.lower().split())
        
        if not q_tokens:
            return 1.0
        
        overlap = len(q_tokens & a_tokens)
        return overlap / len(q_tokens)


class RAGEvaluator:
    """End-to-end RAG evaluation"""
    
    def __init__(self):
        self.retrieval_eval = RetrievalEvaluator()
        self.generation_eval = GenerationEvaluator()
        print("✓ RAG Evaluator initialized")
    
    def evaluate(self, test_cases):
        """Evaluate RAG system on test cases"""
        all_results = []
        
        for case in test_cases:
            result = {"question": case["question"]}
            
            # Retrieval metrics
            if "retrieved_ids" in case and "relevant_ids" in case:
                ret_metrics = self.retrieval_eval.evaluate(
                    case["retrieved_ids"], case["relevant_ids"],
                    case.get("relevance_scores"), k_values=[1, 3, 5]
                )
                result["retrieval"] = ret_metrics
            
            # Generation metrics
            if "generated_answer" in case:
                if "reference_answer" in case:
                    rouge = self.generation_eval.rouge_l(
                        case["generated_answer"], case["reference_answer"]
                    )
                    result["rouge_l"] = rouge
                
                if "context" in case:
                    faith = self.generation_eval.faithfulness(
                        case["generated_answer"], case["context"]
                    )
                    result["faithfulness"] = faith
                
                relevance = self.generation_eval.answer_relevance(
                    case["generated_answer"], case["question"]
                )
                result["answer_relevance"] = relevance
            
            all_results.append(result)
        
        return all_results
    
    def summary(self, results):
        """Print evaluation summary"""
        print(f"\n{'='*60}")
        print(f"📊 RAG Evaluation Summary ({len(results)} test cases)")
        print(f"{'='*60}")
        
        # Aggregate metrics
        metrics_agg = {}
        for r in results:
            for key in ["faithfulness", "answer_relevance"]:
                if key in r:
                    metrics_agg.setdefault(key, []).append(r[key])
            if "rouge_l" in r:
                metrics_agg.setdefault("rouge_l_f1", []).append(r["rouge_l"]["f1"])
            if "retrieval" in r:
                for k, v in r["retrieval"].items():
                    metrics_agg.setdefault(k, []).append(v)
        
        print(f"\n{'Metric':<20} {'Mean':>8} {'Min':>8} {'Max':>8}")
        print("-" * 48)
        for metric, values in sorted(metrics_agg.items()):
            mean_v = np.mean(values)
            min_v = np.min(values)
            max_v = np.max(values)
            print(f"{metric:<20} {mean_v:>8.4f} {min_v:>8.4f} {max_v:>8.4f}")


# === Demo ===
print("=" * 60)
print("RAG Evaluation Demo")
print("=" * 60)

evaluator = RAGEvaluator()

# Create test cases
test_cases = [
    {
        "question": "什么是注意力机制？",
        "retrieved_ids": [3, 1, 7, 2, 5, 8, 4, 6, 9, 10],
        "relevant_ids": [1, 3, 5],
        "relevance_scores": {1: 3, 3: 2, 5: 1, 7: 0, 2: 0},
        "context": "注意力机制是一种让模型关注输入序列中重要部分的技术。它通过计算查询和键的相似度来分配权重。",
        "generated_answer": "注意力机制是一种让模型能够关注输入中重要部分的技术，通过计算相似度来分配不同的权重。",
        "reference_answer": "注意力机制允许模型在处理序列时关注最相关的部分，通过查询-键-值的计算来实现。",
    },
    {
        "question": "Transformer有什么优势？",
        "retrieved_ids": [2, 5, 1, 8, 3, 6, 4, 7, 9, 10],
        "relevant_ids": [2, 5, 8],
        "relevance_scores": {2: 3, 5: 2, 8: 1, 1: 0, 3: 0},
        "context": "Transformer的主要优势包括并行计算能力强、能捕获长距离依赖、可扩展性好。",
        "generated_answer": "Transformer的优势在于并行计算能力强，能够捕获长距离依赖关系，并且具有良好的可扩展性。",
        "reference_answer": "Transformer相比RNN的优势是支持并行训练、处理长序列更有效、扩展性更好。",
    },
    {
        "question": "如何训练BERT模型？",
        "retrieved_ids": [4, 6, 2, 1, 9, 3, 7, 5, 8, 10],
        "relevant_ids": [4, 6, 9],
        "relevance_scores": {4: 3, 6: 2, 9: 1, 2: 0, 1: 0},
        "context": "BERT使用掩码语言模型和下一句预测两个预训练任务。训练数据包括大规模文本语料。",
        "generated_answer": "BERT通过掩码语言模型预训练任务来训练，使用大规模文本语料作为训练数据。",
        "reference_answer": "BERT的训练包括两个阶段：预训练（MLM和NSP任务）和微调（下游任务）。",
    },
]

results = evaluator.evaluate(test_cases)
evaluator.summary(results)

# Show per-case details
print(f"\n--- Per-Case Details ---")
for i, r in enumerate(results):
    print(f"\nQ{i+1}: {r['question']}")
    if "retrieval" in r:
        print(f"  Recall@3: {r['retrieval']['Recall@3']:.2f}, "
              f"MRR: {r['retrieval']['MRR']:.2f}, "
              f"NDCG@5: {r['retrieval'].get('NDCG@5', 0):.4f}")
    print(f"  ROUGE-L F1: {r.get('rouge_l', {}).get('f1', 0):.4f}")
    print(f"  Faithfulness: {r.get('faithfulness', 0):.4f}")
    print(f"  Relevance: {r.get('answer_relevance', 0):.4f}")

print("\n✅ RAG evaluation framework complete!")

## 9. 高级RAG技术与生产部署

### 9.1 高级RAG范式

| 技术 | 原理 | 优势 | 适用场景 |
|------|------|------|----------|
| **Self-RAG** | 模型自主决定是否检索 | 减少不必要检索，降低延迟 | 混合问答（简单+复杂） |
| **Iterative RAG** | 多轮检索，逐步深入 | 处理多跳推理问题 | 复杂问答、研究分析 |
| **Corrective RAG** | 检索后验证和纠正 | 提高答案可靠性 | 高准确性要求场景 |
| **Agentic RAG** | 结合工具使用和检索 | 灵活处理多种任务 | 复杂工作流 |

### 查询扩展技术

```
原始查询: "transformer的注意力机制"

1. 同义词扩展:
   → "self-attention mechanism"
   → "自注意力"

2. 子查询分解:
   → "什么是transformer"
   → "注意力机制的原理"

3. HyDE (假设文档嵌入):
   → 生成假设答案，用其嵌入检索
   → "Transformer的注意力机制是一种..."
```

### 生产向量数据库选型

| 数据库 | 类型 | 规模 | 特点 |
|--------|------|------|------|
| **FAISS** | 库 | <1M | 高性能，内存索引，Meta开源 |
| **Milvus** | 分布式 | >1B | 云原生，支持混合搜索 |
| **Qdrant** | 服务 | <100M | Rust实现，过滤性能好 |
| **Pinecone** | 托管 | 任意 | 全托管，简单易用 |
| **Weaviate** | 服务 | <100M | GraphQL API，模块化 |
| **ChromaDB** | 嵌入式 | <1M | 轻量级，适合原型 |

In [ ]:
# 🎯 Comprehensive Example: Advanced RAG Techniques
# Goal: Implement query expansion, Self-RAG, and production considerations
# Expected outcome: Advanced retrieval techniques with production-ready patterns

class QueryExpander:
    """Expand queries for better retrieval coverage"""
    
    def __init__(self):
        # Synonym dictionary (simplified)
        self.synonyms = {
            "机器学习": ["ML", "machine learning", "统计学习"],
            "深度学习": ["DL", "deep learning", "神经网络"],
            "自然语言处理": ["NLP", "文本处理", "语言理解"],
            "transformer": ["注意力机制", "自注意力", "BERT", "GPT"],
            "训练": ["学习", "优化", "拟合"],
            "性能": ["效果", "表现", "准确率"],
        }
        print("✓ Query Expander initialized")
    
    def expand_synonyms(self, query):
        """Expand query with synonyms"""
        expanded_terms = [query]
        query_lower = query.lower()
        
        for term, syns in self.synonyms.items():
            if term in query_lower or term.lower() in query_lower:
                expanded_terms.extend(syns[:2])  # Add top 2 synonyms
        
        return expanded_terms
    
    def generate_sub_queries(self, query):
        """Break complex query into sub-queries"""
        # Simple heuristic: split on conjunctions and question patterns
        sub_queries = [query]
        
        # Split on "和", "以及", "还有"
        for sep in ["和", "以及", "还有", " and ", " or "]:
            if sep in query:
                parts = query.split(sep)
                sub_queries.extend([p.strip() for p in parts if len(p.strip()) > 3])
        
        return sub_queries
    
    def hypothetical_document(self, query):
        """Generate hypothetical document embedding (HyDE-style)"""
        # In production, use LLM to generate hypothetical answer
        # Here we simulate with template
        templates = [
            f"关于{query}的详细解释如下：这是一个重要的概念...",
            f"{query}是指在机器学习领域中...",
            f"要理解{query}，首先需要了解其基本原理...",
        ]
        return templates[0]
    
    def expand(self, query, method="all"):
        """Full query expansion"""
        results = {"original": query, "expanded": []}
        
        if method in ("all", "synonyms"):
            synonyms = self.expand_synonyms(query)
            results["synonyms"] = synonyms
            results["expanded"].extend(synonyms)
        
        if method in ("all", "sub_queries"):
            sub_queries = self.generate_sub_queries(query)
            results["sub_queries"] = sub_queries
            results["expanded"].extend(sub_queries)
        
        if method in ("all", "hyde"):
            hyde_doc = self.hypothetical_document(query)
            results["hyde"] = hyde_doc
            results["expanded"].append(hyde_doc)
        
        # Deduplicate
        results["expanded"] = list(dict.fromkeys(results["expanded"]))
        return results


class SelfRAG:
    """Self-RAG: Model decides when to retrieve"""
    
    def __init__(self, knowledge_base, embedder, threshold=0.6):
        self.knowledge_base = knowledge_base
        self.embedder = embedder
        self.retrieval_threshold = threshold
        self.stats = {"total_queries": 0, "retrievals_triggered": 0, "direct_answers": 0}
        print(f"✓ Self-RAG initialized (threshold={threshold})")
    
    def needs_retrieval(self, query):
        """Decide if retrieval is needed based on query complexity"""
        # Heuristic signals for retrieval need
        retrieval_signals = {
            "factual_keywords": ["什么是", "定义", "解释", "how", "what", "when", "数据", "统计"],
            "specific_keywords": ["具体", "详细", "步骤", "方法", "算法", "公式"],
            "comparison_keywords": ["区别", "对比", "vs", "比较", "优缺点"],
        }
        
        score = 0.0
        query_lower = query.lower()
        
        for category, keywords in retrieval_signals.items():
            for kw in keywords:
                if kw in query_lower:
                    score += 0.2
        
        # Cap at 1.0
        score = min(score, 1.0)
        return score >= self.retrieval_threshold, score
    
    def process(self, query):
        """Process query with self-retrieval decision"""
        self.stats["total_queries"] += 1
        
        needs_ret, confidence = self.needs_retrieval(query)
        
        result = {
            "query": query,
            "needs_retrieval": needs_ret,
            "retrieval_confidence": confidence,
            "retrieved_docs": [],
            "answer_source": "direct" if not needs_ret else "rag"
        }
        
        if needs_ret:
            self.stats["retrievals_triggered"] += 1
            # Simulate retrieval
            query_emb = self.embedder.encode(query)
            docs = self.knowledge_base.search(query_emb, k=3)
            result["retrieved_docs"] = docs
            result["answer"] = f"[RAG] Based on {len(docs)} retrieved documents..."
        else:
            self.stats["direct_answers"] += 1
            result["answer"] = f"[Direct] Answering from model knowledge..."
        
        return result
    
    def get_stats(self):
        """Get retrieval statistics"""
        total = self.stats["total_queries"]
        if total == 0:
            return self.stats
        retrieval_rate = self.stats["retrievals_triggered"] / total * 100
        self.stats["retrieval_rate"] = f"{retrieval_rate:.1f}%"
        return self.stats


class ProductionRAGConfig:
    """Production RAG system configuration and best practices"""
    
    @staticmethod
    def get_config(scale="medium"):
        """Get recommended configuration for different scales"""
        configs = {
            "small": {
                "description": "小规模 (<10K文档)",
                "vector_db": "FAISS (in-memory)",
                "embedding_model": "all-MiniLM-L6-v2",
                "chunk_size": 256,
                "chunk_overlap": 50,
                "top_k": 3,
                "reranking": False,
                "cache": "LRU (in-memory)",
                "estimated_latency": "50-100ms",
                "estimated_cost": "$50-100/month",
            },
            "medium": {
                "description": "中规模 (10K-1M文档)",
                "vector_db": "Milvus / Qdrant",
                "embedding_model": "BGE-large / E5-large",
                "chunk_size": 512,
                "chunk_overlap": 100,
                "top_k": 5,
                "reranking": True,
                "cache": "Redis",
                "estimated_latency": "100-300ms",
                "estimated_cost": "$200-500/month",
            },
            "large": {
                "description": "大规模 (>1M文档)",
                "vector_db": "Pinecone / Weaviate (managed)",
                "embedding_model": "BGE-large + fine-tuned",
                "chunk_size": 512,
                "chunk_overlap": 128,
                "top_k": 10,
                "reranking": True,
                "cache": "Redis Cluster + CDN",
                "estimated_latency": "200-500ms",
                "estimated_cost": "$1000+/month",
            }
        }
        return configs.get(scale, configs["medium"])
    
    @staticmethod
    def print_checklist():
        """Print production deployment checklist"""
        checklist = {
            "索引管理": [
                "增量索引更新策略",
                "索引版本管理",
                "定期重建索引计划",
                "索引备份和恢复",
            ],
            "性能优化": [
                "查询结果缓存 (Redis/Memcached)",
                "嵌入计算缓存",
                "批量嵌入处理",
                "异步检索流水线",
            ],
            "质量保障": [
                "检索质量监控 (Recall@K)",
                "答案忠实度检查",
                "用户反馈收集",
                "A/B测试框架",
            ],
            "运维监控": [
                "检索延迟监控",
                "向量数据库健康检查",
                "嵌入模型版本管理",
                "数据新鲜度追踪",
            ],
        }
        
        print("📋 Production RAG Deployment Checklist")
        print("=" * 50)
        for category, items in checklist.items():
            print(f"\n📌 {category}")
            for item in items:
                print(f"   □ {item}")


# === Demo: Query Expansion ===
print("=" * 60)
print("Part 1: Query Expansion")
print("=" * 60)

expander = QueryExpander()

queries = [
    "transformer和深度学习的区别",
    "如何提高机器学习模型的性能",
    "什么是自然语言处理",
]

for query in queries:
    result = expander.expand(query)
    print(f"\n📝 Original: {result['original']}")
    print(f"   Synonyms: {result.get('synonyms', [])[:4]}")
    print(f"   Sub-queries: {result.get('sub_queries', [])[:3]}")
    print(f"   Total expanded: {len(result['expanded'])} queries")

# === Demo: Self-RAG ===
print("\n" + "=" * 60)
print("Part 2: Self-RAG Decision Making")
print("=" * 60)

# Create simple knowledge base and embedder for demo
class SimpleEmbedder:
    def encode(self, text):
        np.random.seed(hash(text) % 2**32)
        return np.random.randn(64).astype(np.float32)

class SimpleKB:
    def search(self, query_emb, k=3):
        return [{"text": f"Document {i}", "score": 0.9 - i*0.1} for i in range(k)]

self_rag = SelfRAG(SimpleKB(), SimpleEmbedder(), threshold=0.4)

test_queries = [
    "什么是注意力机制的具体算法步骤？",  # Should trigger retrieval
    "你好",                                # Should not trigger
    "对比BERT和GPT的区别是什么？",         # Should trigger
    "谢谢",                                # Should not trigger
    "解释transformer的数据处理方法",        # Should trigger
]

for query in test_queries:
    result = self_rag.process(query)
    status = "🔍 RETRIEVE" if result["needs_retrieval"] else "💬 DIRECT"
    print(f"  {status} (conf={result['retrieval_confidence']:.1f}): {query}")

stats = self_rag.get_stats()
print(f"\n📊 Stats: {stats['total_queries']} queries, "
      f"{stats['retrievals_triggered']} retrievals ({stats['retrieval_rate']})")

# === Demo: Production Config ===
print("\n" + "=" * 60)
print("Part 3: Production Configuration")
print("=" * 60)

for scale in ["small", "medium", "large"]:
    config = ProductionRAGConfig.get_config(scale)
    print(f"\n📦 {config['description']}")
    print(f"   Vector DB: {config['vector_db']}")
    print(f"   Embedding: {config['embedding_model']}")
    print(f"   Chunk: {config['chunk_size']} tokens (overlap: {config['chunk_overlap']})")
    print(f"   Top-K: {config['top_k']}, Reranking: {config['reranking']}")
    print(f"   Latency: {config['estimated_latency']}, Cost: {config['estimated_cost']}")

print()
ProductionRAGConfig.print_checklist()

print("\n✅ Advanced RAG techniques and production guide complete!")

### 9.2 常见问题

#### Q1: 应该选择哪个嵌入模型？

**A:**
- 中文场景：BGE-large-zh, M3E-large
- 英文场景：E5-large-v2, GTE-large
- 多语言：multilingual-e5-large
- 关键：在你的数据上评估，不要只看排行榜

#### Q2: 检索多少个文档块合适？

**A:**
- 一般推荐：3-5个
- 取决于上下文窗口大小和文档相关性
- 太少：可能遗漏关键信息
- 太多：引入噪声，增加延迟和成本
- 建议：用评估指标（Recall@K）确定最优K值

#### Q3: 如何处理长文档？

**A:**
- 分块：按语义边界切分（段落、章节）
- 重叠：保持10-20%重叠避免信息丢失
- 层次化：先检索文档，再检索段落
- 摘要：为每个文档生成摘要辅助检索

#### Q4: RAG vs 微调，如何选择？

**A:**

| 场景 | 推荐方案 |
|------|---------|
| 知识频繁更新 | RAG（无需重训练） |
| 需要引用来源 | RAG（天然支持溯源） |
| 特定领域风格 | 微调（学习领域语言） |
| 复杂推理任务 | 微调 + RAG |
| 成本敏感 | RAG（无训练成本） |

#### Q5: 如何减少RAG中的幻觉？

**A:**
- 提高检索质量（混合检索 + 重排序）
- 上下文压缩（去除无关信息）
- 忠实度检查（验证答案是否基于上下文）
- 明确指令（"仅基于提供的上下文回答"）
- 设置置信度阈值（低置信度时拒绝回答）

### 9.3 本章总结

#### 核心要点

1. **RAG基础**
   - RAG = 检索 + 增强 + 生成
   - 解决幻觉问题，提供知识溯源
   - 比纯生成更可靠、更可控

2. **向量检索**
   - 稠密嵌入捕获语义相似性
   - FAISS提供高效近似最近邻搜索
   - 余弦相似度是标准度量

3. **文档处理**
   - 分块策略影响检索质量
   - 重叠分块保持上下文连续性
   - 元数据过滤提高检索精度

4. **检索策略**
   - 混合检索（稠密+稀疏）效果最佳
   - 重排序显著提升精度
   - BM25在精确匹配上仍有优势

5. **RAG流水线**
   - 上下文压缩减少噪声
   - 引用生成增加可信度
   - Prompt工程是关键

6. **评估体系**
   - 检索评估：Recall@K, MRR, NDCG
   - 生成评估：ROUGE, 忠实度
   - 端到端评估：答案相关性

7. **高级技术**
   - 查询扩展提升召回率
   - Self-RAG按需检索
   - 生产部署需要缓存和监控

#### 下一步

- **02_agent_systems.ipynb**: AI Agent系统（工具使用、规划、多Agent协作）

---

**Module 8 进度**: 1/? notebooks 完成

## 思考题参考答案

本章深入学习了 RAG（检索增强生成）系统的核心原理与工程实践。以下思考题围绕 RAG 的关键技术节点展开，帮助深化理解。

---

### 问题 1：为什么 RAG 能有效减少 LLM 的"幻觉"（Hallucination）问题？它能彻底消除幻觉吗？

**答：**

RAG 减少幻觉的核心机制是**将生成过程锚定在真实检索到的文档上**。

**RAG 减少幻觉的原因：**

1. **基于事实的生成**：LLM 的输出以检索到的文档为参考，而非完全依赖训练时记忆的知识，降低了"凭空捏造"的概率。
2. **提供可验证上下文**：Prompt 中包含原始文档片段，LLM 有更明确的信息来源，而不是从参数记忆中"猜测"答案。
3. **引用机制**：RAG 流水线可以要求 LLM 在回答中附上来源文档编号，使答案可追溯验证。

**RAG 不能彻底消除幻觉的原因：**

| 残留问题 | 说明 |
|---------|------|
| **检索失败** | 当相关文档未被检索到时，LLM 仍可能依赖自身知识回答，产生幻觉 |
| **文档内容错误** | 若文档库本身包含错误信息，RAG 会"忠实地"生成错误答案 |
| **LLM 过度发挥** | LLM 可能在文档基础上进行不当推断，超出文档内容范围 |
| **上下文窗口限制** | 文档过长时被截断，LLM 可能对缺失部分进行猜测 |

**结论**：RAG 显著降低了幻觉率，但并不能彻底消除。要进一步抑制幻觉，需结合忠实度（Faithfulness）评估、引用验证（Citation Grounding）和 Self-RAG 等技术。

---

### 问题 2：文档分块策略（Chunking Strategy）如何影响 RAG 系统性能？不同场景下应如何选择分块大小？

**答：**

分块策略是 RAG 性能的核心决定因素之一，影响检索精度和生成质量。

**分块大小的权衡：**

$$\text{检索精度} \propto \frac{1}{\text{chunk\_size}} \quad , \quad \text{上下文完整性} \propto \text{chunk\_size}$$

| 分块方式 | 优点 | 缺点 | 适用场景 |
|---------|------|------|---------|
| **小块（128-256 tokens）** | 检索精准，噪声少 | 上下文不完整，可能截断语义 | 精确问答、FAQ |
| **大块（512-1024 tokens）** | 上下文完整，语义连贯 | 检索时引入无关内容 | 长文档理解、报告分析 |
| **重叠分块（Sliding Window）** | 避免边界信息丢失 | 索引体积增大、检索重复 | 通用场景推荐 |
| **语义分块（Semantic Chunking）** | 按语义边界分割，自然流畅 | 计算成本高 | 高质量生产系统 |

**场景选择建议：**
- **技术文档 / FAQ**：小块（256 tokens），精准匹配
- **法律/医疗长文档**：大块（1024 tokens）+ 重叠（overlap=128）
- **多语言场景**：语义分块，避免词语边界问题
- **生产推荐**：512 tokens + 128 tokens 重叠，结合元数据过滤

---

### 问题 3：稠密检索（Dense Retrieval）、稀疏检索（BM25）和混合检索各有什么优缺点？混合检索如何融合两者的分数？

**答：**

**三种检索方式对比：**

| 维度 | BM25（稀疏）| 稠密检索 | 混合检索 |
|-----|------------|---------|---------|
| **原理** | 词频-逆文档频率 | 嵌入向量余弦相似度 | BM25 + 稠密检索融合 |
| **优点** | 速度快、可解释、关键词精确匹配 | 语义理解强、处理同义词 | 兼顾两者优势 |
| **缺点** | 词汇鸿沟（vocabulary mismatch）| 计算成本高、冷启动问题 | 融合权重难调整 |
| **适合** | 关键词明确的查询 | 语义相近的模糊查询 | 通用生产场景 |

**混合检索分数融合（RRF 算法）：**

混合检索常用**倒数排名融合（Reciprocal Rank Fusion, RRF）**：

$$\text{RRF\_score}(d) = \sum_{r \in R} \frac{1}{k + r(d)}$$

其中 $r(d)$ 是文档 $d$ 在排名列表 $r$ 中的名次，$k=60$ 为平滑常数。

也可用**线性加权**：
$$\text{score}(d) = \alpha \cdot \text{dense\_score}(d) + (1-\alpha) \cdot \text{bm25\_score}(d)$$

通常 $\alpha=0.7$ 对稠密检索赋予更大权重。

**实际建议**：混合检索 + 重排序（Cross-encoder Reranker）是目前 RAG 系统的最佳实践，能显著提升 Recall@K 和 NDCG。

---

### 问题 4：RAG 系统的评估指标体系如何设计？Recall@K、NDCG、ROUGE 和忠实度分别衡量什么？

**答：**

RAG 评估需要分别评估**检索质量**和**生成质量**两个维度：

**检索质量指标：**

$$\text{Recall@K} = \frac{|\text{相关文档} \cap \text{Top-K检索结果}|}{|\text{相关文档}|}$$

$$\text{NDCG@K} = \frac{DCG@K}{IDCG@K}, \quad DCG@K = \sum_{i=1}^{K} \frac{rel_i}{\log_2(i+1)}$$

- **Recall@K**：Top-K 结果中召回了多少相关文档，衡量检索覆盖率
- **NDCG@K**：考虑位置权重的排序质量，排名越靠前的相关文档得分越高

**生成质量指标：**

| 指标 | 衡量内容 | 计算方式 |
|-----|---------|---------|
| **ROUGE-L** | 生成答案与参考答案的文本重叠度 | 最长公共子序列（LCS）的 F1 |
| **忠实度（Faithfulness）** | 答案是否仅基于检索文档，不引入外部知识 | NLI 模型判断蕴含关系 |
| **答案相关性（Answer Relevance）** | 答案是否回答了用户的问题 | 嵌入相似度或 LLM 评分 |
| **上下文精度（Context Precision）** | 检索文档中有多少是实际有用的 | 标注或 LLM 自动评估 |

**RAGAS 评估框架**结合了以上多个指标，提供端到端的 RAG 质量评分，是目前最常用的 RAG 评估工具之一。

---

### 问题 5：什么是 Self-RAG？它与标准 RAG 相比有什么核心改进？查询扩展（Query Expansion）又解决了什么问题？

**答：**

**Self-RAG 的核心思想：**

标准 RAG 对每个查询都无差别地执行检索，而 Self-RAG（Self-Reflective RAG）让 LLM 自主决定**何时检索、检索什么、是否使用检索结果**。

Self-RAG 引入了四类特殊 token：

| Token | 含义 | 作用 |
|-------|------|------|
| `[Retrieve]` | 是否需要检索 | 避免不必要的检索开销 |
| `[IsREL]` | 文档是否相关 | 过滤低质量检索结果 |
| `[IsSUP]` | 答案是否有文档支持 | 衡量忠实度 |
| `[IsUSE]` | 答案是否有用 | 端到端质量评估 |

**Self-RAG vs 标准 RAG 对比：**

```
标准 RAG：用户查询 → 必须检索 → 生成答案
Self-RAG：用户查询 → [Retrieve=Yes/No] → 选择性检索 → 验证相关性 → 生成 → 自我批评
```

**查询扩展（Query Expansion）解决的问题：**

原始用户查询往往过于简短或模糊，导致检索召回率低。查询扩展通过以下方式改善检索：

1. **同义词扩展**：将查询词替换或补充同义词
2. **假设文档嵌入（HyDE）**：先让 LLM 生成一个假设性答案，再用该答案的嵌入去检索
3. **多查询生成（Multi-Query）**：LLM 生成多个语义相近的查询，取并集检索结果

$$\text{扩展后Recall} > \text{原始Recall}，但需要权衡精度的下降$$

**总结**：Self-RAG 提升了 RAG 的**自适应性和可靠性**；查询扩展提升了**检索覆盖率**，两者从不同角度优化了 RAG 系统性能。
